# Face Mask Detection System - Training Prototype

**Project Title:** Face Mask Detection System  
**Group ID:** F25PROJECT188B3  
**Author:** BC210400872  
**Supervisor:** Madiha Faqir Hussain

---

## Project Overview

This notebook implements a Convolutional Neural Network (CNN) for binary classification of face images into two categories:
- **With Mask** (wearing face mask)
- **Without Mask** (not wearing face mask)

The prototype covers the complete machine learning pipeline:
1. Dataset loading
2. Data visualization
3. Image preprocessing and augmentation
4. CNN model architecture
5. Model training
6. Performance evaluation
7. Model saving

---
## 1. Import Libraries

Import all necessary libraries for deep learning model development.

In [ ]:
# Deep Learning Framework
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.layers import RandomRotation, RandomFlip, RandomZoom, Rescaling

# Data Handling and Visualization
import numpy as np
import matplotlib.pyplot as plt

# System utilities
import os
from pathlib import Path

# Display versions for reproducibility
print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print(f"NumPy Version: {np.__version__}")

---
## 2. Load Dataset

Load the face mask dataset using TensorFlow's efficient data loading utilities.

**Dataset Information:**
- Location: `./dataset`
- Total Images: 1,376
- Classes: with_mask (690 images), without_mask (686 images)
- Format: JPG images

In [ ]:
# Define dataset path
dataset_path = './dataset'

# Define image parameters
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
SEED = 42  # For reproducibility

# Load training dataset (80% of total images)
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,  # Reserve 20% for validation
    subset="training",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='binary'  # Binary classification (0 or 1)
)

# Load validation dataset (20% of total images)
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# Get class names
class_names = train_dataset.class_names
print(f"\nClass Names: {class_names}")
print(f"Class Encoding: {class_names[0]} = 0, {class_names[1]} = 1")

In [ ]:
# Display dataset information
print("\n" + "="*60)
print("DATASET INFORMATION")
print("="*60)

# Calculate total batches and images
num_train_batches = tf.data.experimental.cardinality(train_dataset).numpy()
num_val_batches = tf.data.experimental.cardinality(validation_dataset).numpy()

# Estimate total images (batches * batch_size, approximately)
approx_train_images = num_train_batches * BATCH_SIZE
approx_val_images = num_val_batches * BATCH_SIZE

print(f"\nTraining Set:")
print(f"  - Number of batches: {num_train_batches}")
print(f"  - Approximate images: {approx_train_images}")

print(f"\nValidation Set:")
print(f"  - Number of batches: {num_val_batches}")
print(f"  - Approximate images: {approx_val_images}")

# Display shape of a sample batch
for images, labels in train_dataset.take(1):
    print(f"\nBatch Shape:")
    print(f"  - Images shape: {images.shape}")
    print(f"    (Batch Size, Height, Width, Channels)")
    print(f"  - Labels shape: {labels.shape}")
    print(f"  - Image dtype: {images.dtype}")
    print(f"  - Pixel value range: [{images.numpy().min():.1f}, {images.numpy().max():.1f}]")

print("\n" + "="*60)

---
## 3. Data Visualization

Visualize sample images from the training dataset to verify proper loading and understand the data.

In [ ]:
# Extract a batch of images and labels for visualization
sample_images, sample_labels = next(iter(train_dataset))

# Create a figure with 2 rows and 5 columns (10 sample images)
plt.figure(figsize=(15, 6))
plt.suptitle('Sample Images from Training Dataset', fontsize=16, fontweight='bold')

for i in range(10):
    plt.subplot(2, 5, i + 1)
    
    # Display image (convert to uint8 for proper display)
    plt.imshow(sample_images[i].numpy().astype("uint8"))
    
    # Get label and convert to class name
    label_value = int(sample_labels[i].numpy())
    label_text = class_names[label_value]
    
    # Format label for display
    display_label = "With Mask" if label_text == "with_mask" else "Without Mask"
    
    # Set color based on class (green for with mask, red for without mask)
    color = 'green' if label_text == "with_mask" else 'red'
    
    plt.title(display_label, fontsize=12, fontweight='bold', color=color)
    plt.axis('off')

plt.tight_layout()
plt.show()

print("\nVisualization Complete: 10 sample images displayed with their labels.")

---
## 4. Data Preprocessing & Augmentation

### Preprocessing Steps:
1. **Normalization**: Scale pixel values from [0, 255] to [0, 1]
2. **Data Augmentation**: Apply random transformations to increase model generalization

### Why Data Augmentation?
Data augmentation artificially expands the training dataset by creating modified versions of images. This helps:
- Prevent overfitting
- Improve model generalization
- Make the model robust to variations in real-world scenarios

In [ ]:
# Create data augmentation pipeline
# These layers will be integrated into the model and only applied during training
data_augmentation = Sequential([
    RandomRotation(0.2),      # Rotate images by up to 20% (±72 degrees)
    RandomFlip("horizontal"),  # Randomly flip images horizontally
    RandomZoom(0.1)           # Randomly zoom in/out by 10%
], name="data_augmentation")

print("Data Augmentation Pipeline Created:")
print("  - Random Rotation: ±20%")
print("  - Random Horizontal Flip")
print("  - Random Zoom: ±10%")

In [ ]:
# Visualize effect of data augmentation on a sample image
plt.figure(figsize=(15, 3))
plt.suptitle('Data Augmentation Examples (Same Image)', fontsize=14, fontweight='bold')

# Take one image
sample_image = sample_images[0:1]  # Keep batch dimension

# Show original
plt.subplot(1, 6, 1)
plt.imshow(sample_image[0].numpy().astype("uint8"))
plt.title("Original", fontweight='bold')
plt.axis('off')

# Show 5 augmented versions
for i in range(5):
    augmented_image = data_augmentation(sample_image, training=True)
    plt.subplot(1, 6, i + 2)
    plt.imshow(augmented_image[0].numpy().astype("uint8"))
    plt.title(f"Augmented {i+1}", fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Configure dataset for performance optimization
# AUTOTUNE allows TensorFlow to dynamically adjust buffer sizes for optimal performance
AUTOTUNE = tf.data.AUTOTUNE

# Apply caching and prefetching for faster training
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

print("Performance Optimization Applied:")
print("  - cache(): Images kept in memory after first epoch")
print("  - prefetch(): Next batch prepared while current batch is being processed")
print("\nDatasets are ready for training!")

---
## 5. CNN Model Architecture

### Why Convolutional Neural Networks (CNN)?
CNNs are specifically designed for image processing tasks because they:
- Automatically learn spatial hierarchies of features
- Use shared weights to reduce parameters (efficient)
- Are translation-invariant (detect features anywhere in the image)

### Model Architecture:
- **Input:** 224×224×3 RGB images
- **Convolutional Layers:** Extract features (edges, textures, patterns)
- **MaxPooling Layers:** Reduce spatial dimensions and computational cost
- **Flatten Layer:** Convert 2D feature maps to 1D vector
- **Dense Layers:** Learn complex patterns for classification
- **Output:** Single neuron with sigmoid activation (binary classification)

In [ ]:
# Build the CNN model
model = Sequential([
    # Input layer - normalize pixel values from [0, 255] to [0, 1]
    Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    
    # Data augmentation (only active during training)
    data_augmentation,
    
    # First Convolutional Block
    # 32 filters of size 3x3, ReLU activation, same padding preserves dimensions
    Conv2D(32, kernel_size=3, activation='relu', padding='same', name='conv1'),
    MaxPooling2D(pool_size=2, name='pool1'),  # Reduces spatial dimensions by half
    
    # Second Convolutional Block
    # 64 filters - learns more complex features
    Conv2D(64, kernel_size=3, activation='relu', padding='same', name='conv2'),
    MaxPooling2D(pool_size=2, name='pool2'),
    
    # Third Convolutional Block
    # 128 filters - learns even more abstract features
    Conv2D(128, kernel_size=3, activation='relu', padding='same', name='conv3'),
    MaxPooling2D(pool_size=2, name='pool3'),
    
    # Flatten - convert 3D feature maps to 1D feature vector
    Flatten(name='flatten'),
    
    # Fully Connected Layer
    Dense(128, activation='relu', name='dense1'),
    Dropout(0.5, name='dropout'),  # Randomly drop 50% of connections to prevent overfitting
    
    # Output Layer
    # Single neuron with sigmoid activation outputs probability [0, 1]
    # 0 = with_mask, 1 = without_mask (or vice versa based on class order)
    Dense(1, activation='sigmoid', name='output')
], name='FaceMaskDetectionCNN')

print("CNN Model Architecture Created Successfully!\n")

In [ ]:
# Compile the model
# Adam optimizer: Adaptive learning rate, efficient for most problems
# Binary crossentropy: Standard loss function for binary classification
# Accuracy metric: Percentage of correct predictions

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model Compiled with:")
print("  - Optimizer: Adam (learning_rate=0.001)")
print("  - Loss Function: Binary Crossentropy")
print("  - Metrics: Accuracy")
print("\n" + "="*60)

---
## 10. Save Training Results

Save all important results including plots, metrics, and model for FYP submission.

In [ ]:
# Create a results folder to organize all outputs
results_folder = 'training_results'
os.makedirs(results_folder, exist_ok=True)

# Ensure all necessary variables are defined (in case cells run out of order)
if 'approx_train_images' not in globals():
    num_train_batches = tf.data.experimental.cardinality(train_dataset).numpy()
    num_val_batches = tf.data.experimental.cardinality(validation_dataset).numpy()
    approx_train_images = num_train_batches * BATCH_SIZE
    approx_val_images = num_val_batches * BATCH_SIZE

print(f"Creating results folder: {results_folder}")
print("="*60)

In [ ]:
# Save training history plots as high-quality images

# Extract training history (in case this cell is run separately)
train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
train_loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, EPOCHS + 1)

# Recreate the plots with higher DPI for better quality
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
fig.suptitle('Training and Validation Metrics', fontsize=16, fontweight='bold')

# Plot 1: Accuracy
ax1.plot(epochs_range, train_accuracy, 'b-', label='Training Accuracy', linewidth=2)
ax1.plot(epochs_range, val_accuracy, 'r-', label='Validation Accuracy', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)
ax1.set_xticks(epochs_range)

# Plot 2: Loss
ax2.plot(epochs_range, train_loss, 'b-', label='Training Loss', linewidth=2)
ax2.plot(epochs_range, val_loss, 'r-', label='Validation Loss', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(epochs_range)

plt.tight_layout()

# Save the plot
plot_filename = f'{results_folder}/training_history.png'
plt.savefig(plot_filename, dpi=150, bbox_inches='tight')
print(f"✓ Saved training history plot: {plot_filename}")

plt.show()

In [ ]:
# Save sample images visualization

# Recreate sample images plot
plt.figure(figsize=(15, 6), dpi=150)
plt.suptitle('Sample Images from Training Dataset', fontsize=16, fontweight='bold')

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(sample_images[i].numpy().astype("uint8"))
    
    label_value = int(sample_labels[i].numpy())
    label_text = class_names[label_value]
    display_label = "With Mask" if label_text == "with_mask" else "Without Mask"
    color = 'green' if label_text == "with_mask" else 'red'
    
    plt.title(display_label, fontsize=12, fontweight='bold', color=color)
    plt.axis('off')

plt.tight_layout()

# Save the plot
sample_images_filename = f'{results_folder}/sample_images.png'
plt.savefig(sample_images_filename, dpi=150, bbox_inches='tight')
print(f"✓ Saved sample images: {sample_images_filename}")

plt.show()

In [ ]:
# Save data augmentation examples

plt.figure(figsize=(15, 3), dpi=150)
plt.suptitle('Data Augmentation Examples (Same Image)', fontsize=14, fontweight='bold')

# Show original
plt.subplot(1, 6, 1)
plt.imshow(sample_images[0].numpy().astype("uint8"))
plt.title("Original", fontweight='bold')
plt.axis('off')

# Show 5 augmented versions
for i in range(5):
    augmented_image = data_augmentation(sample_images[0:1], training=True)
    plt.subplot(1, 6, i + 2)
    plt.imshow(augmented_image[0].numpy().astype("uint8"))
    plt.title(f"Augmented {i+1}", fontweight='bold')
    plt.axis('off')

plt.tight_layout()

# Save the plot
augmentation_filename = f'{results_folder}/data_augmentation.png'
plt.savefig(augmentation_filename, dpi=150, bbox_inches='tight')
print(f"✓ Saved data augmentation examples: {augmentation_filename}")

plt.show()

In [ ]:
# Save training metrics to CSV file for analysis
import csv

metrics_csv = f'{results_folder}/training_metrics.csv'

with open(metrics_csv, 'w', newline='') as file:
    writer = csv.writer(file)
    
    # Write header
    writer.writerow(['Epoch', 'Training_Accuracy', 'Validation_Accuracy', 'Training_Loss', 'Validation_Loss'])
    
    # Write data
    for i in range(len(train_accuracy)):
        writer.writerow([
            i + 1,
            f"{train_accuracy[i]:.6f}",
            f"{val_accuracy[i]:.6f}",
            f"{train_loss[i]:.6f}",
            f"{val_loss[i]:.6f}"
        ])

print(f"✓ Saved training metrics CSV: {metrics_csv}")

In [ ]:
# Save model summary to text file

summary_file = f'{results_folder}/model_summary.txt'

with open(summary_file, 'w') as f:
    # Redirect model.summary() output to file
    model.summary(print_fn=lambda x: f.write(x + '\n'))
    
    # Add additional information
    f.write('\n' + '='*60 + '\n')
    f.write('FINAL TRAINING RESULTS\n')
    f.write('='*60 + '\n\n')
    f.write(f'Training Accuracy: {train_accuracy[-1]:.4f} ({train_accuracy[-1]*100:.2f}%)\n')
    f.write(f'Training Loss: {train_loss[-1]:.4f}\n\n')
    f.write(f'Validation Accuracy: {val_accuracy[-1]:.4f} ({val_accuracy[-1]*100:.2f}%)\n')
    f.write(f'Validation Loss: {val_loss[-1]:.4f}\n\n')
    f.write(f'Total Epochs: {EPOCHS}\n')
    f.write(f'Image Size: {IMG_HEIGHT}x{IMG_WIDTH}\n')
    f.write(f'Batch Size: {BATCH_SIZE}\n')
    f.write(f'Dataset Size: ~{approx_train_images + approx_val_images} images\n')
    f.write(f'Training Images: ~{approx_train_images}\n')
    f.write(f'Validation Images: ~{approx_val_images}\n')

print(f"✓ Saved model summary: {summary_file}")

In [ ]:
# Copy the trained model to results folder
import shutil

model_in_results = f'{results_folder}/facemask_model.h5'
shutil.copy(model_filename, model_in_results)

print(f"✓ Copied model to results folder: {model_in_results}")
print(f"\nModel file size: {os.path.getsize(model_in_results) / (1024*1024):.2f} MB")

In [ ]:
# Create a comprehensive results summary document

summary_doc = f'{results_folder}/RESULTS_SUMMARY.txt'

with open(summary_doc, 'w') as f:
    f.write('='*70 + '\n')
    f.write('FACE MASK DETECTION SYSTEM - TRAINING RESULTS\n')
    f.write('='*70 + '\n\n')
    
    f.write('PROJECT INFORMATION\n')
    f.write('-'*70 + '\n')
    f.write('Project Title: Face Mask Detection System\n')
    f.write('Group ID: F25PROJECT188B3\n')
    f.write('Author: BC210400872\n')
    f.write('Supervisor: Madiha Faqir Hussain\n\n')
    
    f.write('DATASET INFORMATION\n')
    f.write('-'*70 + '\n')
    f.write(f'Total Images: ~{approx_train_images + approx_val_images}\n')
    f.write(f'Training Set: ~{approx_train_images} images (80%)\n')
    f.write(f'Validation Set: ~{approx_val_images} images (20%)\n')
    f.write(f'Classes: {class_names}\n')
    f.write(f'Image Size: {IMG_HEIGHT}x{IMG_WIDTH} pixels\n')
    f.write(f'Image Format: RGB (3 channels)\n\n')
    
    f.write('MODEL ARCHITECTURE\n')
    f.write('-'*70 + '\n')
    f.write('CNN Architecture:\n')
    f.write('  - Input: 224x224x3 RGB images\n')
    f.write('  - Rescaling: Normalize pixels to [0, 1]\n')
    f.write('  - Data Augmentation: Rotation, Flip, Zoom\n')
    f.write('  - Conv2D Layer 1: 32 filters (3x3) + MaxPooling\n')
    f.write('  - Conv2D Layer 2: 64 filters (3x3) + MaxPooling\n')
    f.write('  - Conv2D Layer 3: 128 filters (3x3) + MaxPooling\n')
    f.write('  - Flatten Layer\n')
    f.write('  - Dense Layer: 128 neurons + Dropout (0.5)\n')
    f.write('  - Output Layer: 1 neuron (Sigmoid)\n\n')
    
    f.write('TRAINING CONFIGURATION\n')
    f.write('-'*70 + '\n')
    f.write(f'Epochs: {EPOCHS}\n')
    f.write(f'Batch Size: {BATCH_SIZE}\n')
    f.write('Optimizer: Adam (learning_rate=0.001)\n')
    f.write('Loss Function: Binary Crossentropy\n')
    f.write('Metrics: Accuracy\n\n')
    
    f.write('FINAL RESULTS\n')
    f.write('-'*70 + '\n')
    f.write(f'Training Accuracy:   {train_accuracy[-1]:.4f} ({train_accuracy[-1]*100:.2f}%)\n')
    f.write(f'Training Loss:       {train_loss[-1]:.4f}\n')
    f.write(f'Validation Accuracy: {val_accuracy[-1]:.4f} ({val_accuracy[-1]*100:.2f}%)\n')
    f.write(f'Validation Loss:     {val_loss[-1]:.4f}\n\n')
    
    # Performance assessment
    if val_accuracy[-1] >= 0.95:
        f.write('Performance: EXCELLENT (>95% accuracy)\n')
    elif val_accuracy[-1] >= 0.90:
        f.write('Performance: GOOD (>90% accuracy)\n')
    elif val_accuracy[-1] >= 0.85:
        f.write('Performance: ACCEPTABLE (>85% accuracy)\n')
    else:
        f.write('Performance: NEEDS IMPROVEMENT\n')
    
    f.write('\n' + '='*70 + '\n')
    f.write('DELIVERABLES IN THIS FOLDER\n')
    f.write('='*70 + '\n')
    f.write('1. facemask_model.h5 - Trained CNN model\n')
    f.write('2. training_history.png - Accuracy and loss plots\n')
    f.write('3. sample_images.png - Sample dataset images\n')
    f.write('4. data_augmentation.png - Augmentation examples\n')
    f.write('5. training_metrics.csv - Epoch-by-epoch metrics\n')
    f.write('6. model_summary.txt - Model architecture details\n')
    f.write('7. RESULTS_SUMMARY.txt - This file\n')
    f.write('\n' + '='*70 + '\n')

print(f"✓ Saved results summary: {summary_doc}")
print("\nAll results saved successfully!")

In [ ]:
# Save training metrics to CSV file
import csv

metrics_csv = f'{results_folder}/training_metrics.csv'

with open(metrics_csv, 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Epoch', 'Training_Accuracy', 'Validation_Accuracy', 'Training_Loss', 'Validation_Loss'])
    
    for i in range(len(train_accuracy)):
        writer.writerow([
            i + 1,
            f"{train_accuracy[i]:.6f}",
            f"{val_accuracy[i]:.6f}",
            f"{train_loss[i]:.6f}",
            f"{val_loss[i]:.6f}"
        ])

print(f"✓ Saved training metrics: {metrics_csv}")

In [ ]:
# Save model summary to text file

summary_file = f'{results_folder}/model_summary.txt'

with open(summary_file, 'w') as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))
    
    f.write('\n' + '='*60 + '\n')
    f.write('FINAL TRAINING RESULTS\n')
    f.write('='*60 + '\n\n')
    f.write(f'Training Accuracy: {train_accuracy[-1]:.4f} ({train_accuracy[-1]*100:.2f}%)\n')
    f.write(f'Training Loss: {train_loss[-1]:.4f}\n\n')
    f.write(f'Validation Accuracy: {val_accuracy[-1]:.4f} ({val_accuracy[-1]*100:.2f}%)\n')
    f.write(f'Validation Loss: {val_loss[-1]:.4f}\n\n')
    f.write(f'Total Epochs: {EPOCHS}\n')
    f.write(f'Image Size: {IMG_HEIGHT}x{IMG_WIDTH}\n')
    f.write(f'Batch Size: {BATCH_SIZE}\n')

print(f"✓ Saved model summary: {summary_file}")

In [ ]:
# Copy the trained model to results folder
import shutil

model_in_results = f'{results_folder}/facemask_model.h5'
shutil.copy(model_filename, model_in_results)

print(f"✓ Copied model to results folder")

In [ ]:
# List all files in results folder
print("\n" + "="*60)
print("TRAINING RESULTS - ALL FILES SAVED")
print("="*60)
print(f"\nLocation: {os.path.abspath(results_folder)}\n")

for filename in sorted(os.listdir(results_folder)):
    filepath = os.path.join(results_folder, filename)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  ✓ {filename:35s} ({size_mb:6.2f} MB)")

print("\n" + "="*60)
print("All training results have been saved!")
print("You can use these files for your FYP report and presentation.")
print("="*60)

In [ ]:
# Display model architecture summary
print("\nMODEL ARCHITECTURE SUMMARY")
print("="*60)
model.summary()
print("="*60)

---
## 6. Model Training

Train the CNN model on the training dataset and validate on the validation dataset.

**Training Configuration:**
- Epochs: 15 (can be adjusted based on convergence)
- The model will learn to distinguish between masked and non-masked faces
- Training history is recorded for analysis

In [ ]:
# Define number of epochs
EPOCHS = 15

print("Starting Model Training...")
print(f"Training for {EPOCHS} epochs")
print("This may take 10-15 minutes on CPU or 2-3 minutes on GPU...\n")
print("="*60)

# Train the model
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    verbose=1  # Display training progress
)

print("\n" + "="*60)
print("Training Complete!")
print("="*60)

---
## 7. Training History Visualization

Visualize training and validation metrics over epochs to analyze model performance.

**What to look for:**
- **Good fit:** Training and validation curves are close together
- **Overfitting:** Validation accuracy decreases while training accuracy increases
- **Underfitting:** Both training and validation accuracy remain low
- **Convergence:** Metrics stabilize and stop improving significantly

In [ ]:
# Extract training history
train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
train_loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, EPOCHS + 1)

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training and Validation Metrics', fontsize=16, fontweight='bold')

# Plot 1: Accuracy
ax1.plot(epochs_range, train_accuracy, 'b-', label='Training Accuracy', linewidth=2)
ax1.plot(epochs_range, val_accuracy, 'r-', label='Validation Accuracy', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)
ax1.set_xticks(epochs_range)

# Plot 2: Loss
ax2.plot(epochs_range, train_loss, 'b-', label='Training Loss', linewidth=2)
ax2.plot(epochs_range, val_loss, 'r-', label='Validation Loss', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(epochs_range)

plt.tight_layout()
plt.show()

# Print final epoch metrics
print("\nFinal Training Metrics:")
print(f"  - Training Accuracy: {train_accuracy[-1]:.4f} ({train_accuracy[-1]*100:.2f}%)")
print(f"  - Training Loss: {train_loss[-1]:.4f}")
print(f"\nFinal Validation Metrics:")
print(f"  - Validation Accuracy: {val_accuracy[-1]:.4f} ({val_accuracy[-1]*100:.2f}%)")
print(f"  - Validation Loss: {val_loss[-1]:.4f}")

### Analysis of Training Results

**Interpreting the Plots:**
1. **Accuracy Plot:** Shows how well the model classifies faces over time
   - Both curves should increase and converge
   - If validation accuracy is significantly lower than training, consider more regularization

2. **Loss Plot:** Shows how well the model minimizes prediction error
   - Both curves should decrease and converge
   - If validation loss increases while training loss decreases, this indicates overfitting

---
## 8. Model Evaluation

Evaluate the trained model on the validation dataset to measure final performance.

In [ ]:
# Evaluate model on validation dataset
print("Evaluating Model on Validation Dataset...\n")
print("="*60)

val_loss, val_accuracy = model.evaluate(validation_dataset, verbose=1)

print("="*60)
print("\nFINAL MODEL PERFORMANCE")
print("="*60)
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
print("="*60)

# Provide performance assessment
if val_accuracy >= 0.95:
    print("\n✓ EXCELLENT: Model exceeds 95% accuracy target!")
elif val_accuracy >= 0.90:
    print("\n✓ GOOD: Model achieves >90% accuracy - suitable for prototype.")
elif val_accuracy >= 0.85:
    print("\n⚠ ACCEPTABLE: Model achieves >85% accuracy - consider more training epochs.")
else:
    print("\n⚠ NEEDS IMPROVEMENT: Consider adjusting hyperparameters or adding more data.")

---
## 9. Save Trained Model

Save the trained model in HDF5 format for future use in deployment or inference.

**Model File:** `facemask_model.h5`  
**Format:** HDF5 (Hierarchical Data Format)  
**Contains:** Complete model architecture, trained weights, optimizer state

In [ ]:
# Define model filename
model_filename = 'facemask_model.h5'

# Save the model
model.save(model_filename)

print("="*60)
print("MODEL SAVED SUCCESSFULLY")
print("="*60)
print(f"\nFilename: {model_filename}")
print(f"Location: {os.path.abspath(model_filename)}")

# Display file size
if os.path.exists(model_filename):
    file_size_mb = os.path.getsize(model_filename) / (1024 * 1024)
    print(f"File Size: {file_size_mb:.2f} MB")

print("\nThe model can be loaded later using:")
print("  model = tf.keras.models.load_model('facemask_model.h5')")
print("\n" + "="*60)

---
## Summary & Conclusions

### Project Deliverables Completed:
✓ Dataset loaded and preprocessed (1,376 images)  
✓ Data augmentation pipeline implemented  
✓ CNN model architecture designed and built  
✓ Model trained for binary classification  
✓ Training history visualized and analyzed  
✓ Model evaluated on validation dataset  
✓ Trained model saved as `facemask_model.h5`  

### Key Technical Concepts Demonstrated:
1. **Convolutional Layers:** Feature extraction from images
2. **MaxPooling:** Spatial dimension reduction and computational efficiency
3. **Data Augmentation:** Improving model generalization
4. **Dropout:** Regularization technique to prevent overfitting
5. **Binary Classification:** Using sigmoid activation for two-class problems
6. **Adam Optimizer:** Adaptive learning rate optimization

### Potential Improvements (Future Work):
- Increase training epochs if validation accuracy is still improving
- Implement transfer learning using pre-trained models (MobileNet, VGG16)
- Add more data augmentation techniques (brightness, contrast adjustment)
- Experiment with different CNN architectures
- Implement additional evaluation metrics (Precision, Recall, F1-Score)
- Create confusion matrix for detailed performance analysis

---

**End of Notebook**